# TL-mesh support structure: hyperplane fitting (old)

**Old notebook, recovered from the history and not ported:** it is from before the
sphere/plane swap (spheres on faces) and uses old paths (`Code/data`) and the old viewer setup,
so it does not run as is. Kept as reference for fitting the spheres of an L-mesh to a
hyperplane (linear sphere complex) and drawing the corresponding plane, see *Hyper plane
fitting*. The current version is `lmesh_support_strcutures.ipynb`.


## 0 — Python path

Add `Code/` and `hananLab/` so imports resolve. Adjust `PROJECT_ROOT` if the notebook is moved.

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
for sub in ("", "Code", "hananLab"):  # "" adds PROJECT_ROOT itself (for server/)
    p = os.path.join(PROJECT_ROOT, sub) if sub else PROJECT_ROOT
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", PROJECT_ROOT)

## 1 — Connect to the viewer

In [ ]:
import kayviz as kv

kv.init()   # pings the server; raises if it is not running

## 2 — Load a mesh

In [ ]:
import numpy as np
from hanan.geometry.utils import read_obj, normalize_vertices
from hanan.geometry.mesh import Mesh

DATA = os.path.join(PROJECT_ROOT, "Code", "data")

V, F = read_obj(os.path.join(DATA, "hexMeshTestSP.obj"))
V    = normalize_vertices(V)

print(f"Vertices: {len(V)}   Faces: {len(F)}")

## 3 — Push the mesh and open the viewer

`kv.show()` is non-blocking in Jupyter — the call returns immediately and the viewer tab updates.

In [ ]:
  # opens http://localhost:8000?app=script if not already open

## Construct an L-mesh

In [ ]:
from hanan.geometry.utils import (compute_cone_axes, 
                                  conical_mesh_smallest_curvature_sphere, draw_polygon, unit, orth_proj,
                                  face_planarity)
from LMesh.L_mesh import Lmesh

kv.clear()   # remove anything left from a previous run

kv.register_surface_mesh("Base Mesh", V, F, color=(0.7, 0.7, 0.9), show_edges=True)

kv.show()  

mesh = Mesh()
mesh.make_mesh(V, F)

ng = unit(np.array([-0.5,-0.2,0]))
pg = np.array([0.35,-0.1,0.2])
hg = -ng@pg

aux = ng + (np.array([0, 1, 0]) if np.isclose(np.abs(np.array([1, 0, 0]) @ ng), 1) else np.array([1, 0, 0]))

v1 = unit(orth_proj(aux, ng))
v2 = unit(np.cross(ng, v1))
v1 *= 1
v2 *= 1
vertices = np.array([pg + v1 + v2, pg + v1 - v2, pg - v1 - v2, pg - v1 + v2])

kv.register_surface_mesh("Plane", vertices, [np.arange(len(vertices))[:, None]],
                         color=(0.1, 0.1, 0.1), transparency=0.6)

# plane_normal = np.array((pg, pg + ng))
# kv.register_curve_network("Plane Normal", plane_normal, [0,1], color=(0.9, 0.1, 0.1))


axes, angles = compute_cone_axes(mesh)
_, smallest_radii = conical_mesh_smallest_curvature_sphere(
    mesh.vertices, axes, angles, mesh.vertex_adjacency_list()
)

radii = 0.6 * np.abs(smallest_radii)

theta = np.pi/4.5


for i in range(len(radii)):
    #radii[i] = (mesh.vertices[i] @ ng + hg)/(np.cos(theta) -  (axes[i] @ ng)/(np.sin(angles[i])))
    radii[i] = (mesh.vertices[i] @ ng + hg)/(np.cosh(theta) -  (axes[i] @ ng)/(np.sin(angles[i])))

d = (mesh.vertices + (radii/np.sin(angles))[:, None] * axes)@ng + hg

print("sin(angle)", d/radii) 

centers = mesh.vertices + (radii/np.sin(angles))[:, None] * axes

inn_idx = mesh.inner_vertices()

lmesh = Lmesh()
lmesh.init_L_mesh(
    mesh,
    orientation=1,
    combinatorics="dual",
    mode="SphCongruence",
    radii=radii[inn_idx],
    centers=centers[inn_idx],
)
# lmesh.init_L_mesh(
#     basemesh,
#     orientation=1,
#     combinatorics="dual",
#     mode="SphCongruence",
#     radii=radii[inn_idx],
#     centers=centers[inn_idx],
# )
print( lmesh.compute_oriented_contact_dev()) 

lmesh.improve_contact()


sC, sF = lmesh.sphere_center_mesh()

sF_planarity = face_planarity(sC, sF)

kv.register_surface_mesh("Sphere Centers", sC, sF, color=(0.7, 0.9, 0.7), show_edges=True)

kv.add_scalar_quantity("Sphere Centers", "Planairty",defined_on="faces",  values=sF_planarity)

kv.add_vector_quantity("Base Mesh", "Cone Axis", axes)

print( lmesh.compute_oriented_contact_dev()) 
lmesh.visualize()


In [ ]:
ps_V, ps_F = lmesh.planar_support_structure(0.05)
kv.register_surface_mesh("Planar Support Structure", ps_V, ps_F, color=(0.9, 0.7, 0.7), show_edges=True)

In [ ]:
psudoFocal_v = mesh.vertices + (smallest_radii/np.sin(angles))[:, None] * axes
kv.register_surface_mesh("Psudo Focal Points", psudoFocal_v, mesh.faces, color=(0.7, 0.9, 0.7), show_edges=True)

In [ ]:
lmesh.compute_second_envelope()
lmesh.visualize_second_envelope("se")

In [ ]:
from hanan.geometry.utils import draw_polysphere,  polysphere, merge_meshes
import random

sph_id = random.sample(range(len(lmesh.sf_centers)), 10)

spheres_v, spheres_f = [], []

for si in sph_id:
    sph_V, sph_F = polysphere(lmesh.sf_centers[si], lmesh.sf_radii[si], 4)

    spheres_v.append(sph_V)
    spheres_f.append(sph_F)


set_spheresV, set_spheresF = merge_meshes(spheres_v, spheres_f)

kv.register_surface_mesh("Sampled_spheres", set_spheresV, set_spheresF, color=(0.0, 0.0, 0.9), transparency=0.5)

# Hyper plane fitting

The idea is to fit a hyperplane of the form $H: n.x + h = 0$ where $n = (n_1,n_2,n_3,n_4)$ and $x=(x_1,x_2,x_3,x_4)$. We fit with points that correspond to the or-spheres  $x = (c,r)$. We then verify if $H$ is Euclidean, pseudo-Euclidean or Isotropic by computing $n_1^2 + n_2^2 + n_3^2 - n_4^2$ is $< 0$,  $>0$ or $=0$

In [ ]:

radii_to_fit = 0.8 * smallest_radii
centers_to_fit = mesh.vertices + (radii_to_fit/np.sin(angles))[:, None] * axes

iso_spheres = np.column_stack((centers_to_fit[inn_idx], radii_to_fit[inn_idx]))

import numpy as np
from dataclasses import dataclass
from typing import Literal

from hanan.geometry.utils import draw_plane

@dataclass
class FitResult:
    n: np.ndarray        # normal (n1,n2,n3,n4), unit in Euclidean sense
    h: float             # offset
    kind: Literal['euclidean', 'pseudo_euclidean', 'isotropic']
    signature: float     # n1²+n2²+n3²-n4²
    residuals: np.ndarray

def fit_hyperplane(spheres: list) -> FitResult:
    """
    Fit a hyperplane n.x + h = 0 in Lie sphere space.

    Parameters
    ----------
    spheres : list of spheres, each given as (x,y,z,r) with r the radius

    Returns
    -------
    FitResult with normal n, offset h, kind, signature, residuals
    """
    # Build the 4D point matrix  X : (N, 4)
    X = np.array(spheres, dtype=float)  # (N, 4)
    N = len(X)

    # --- Fit: hyperplane n.x + h = 0 ---
    # Homogenize: augment with a column of ones for the offset h
    # We want to find (n, h) minimizing sum |n.x_i + h|²
    # This is just PCA on the augmented points, find the smallest singular vector

    ones = np.ones((N, 1))
    A = np.hstack([X, ones])            # (N, 5):  columns are (x1,x2,x3,x4,1)

    _, _, Vt = np.linalg.svd(A, full_matrices=True)
    # Smallest singular value → last row of Vt
    v = Vt[-1]                          # (5,)
    n, h = v[:4], v[4]

    # Normalize n to unit Euclidean norm (convention)
    norm = np.linalg.norm(n)
    n, h = n / norm, h / norm

    # --- Classify by Minkowski signature ---
    sig = n[0]**2 + n[1]**2 + n[2]**2 - n[3]**2

    eps = 1e-10
    if np.isclose(abs(sig), 0, atol=eps):
        kind = 'isotropic'
    elif np.sign(sig) < 0:
        kind = 'euclidean'
    else:
        kind = 'pseudo_euclidean'

    angle = np.arccos( np.abs(n[3]/np.linalg.norm(n) ))

    if angle < np.pi/4:
        print("Euclidean")
    elif angle > np.pi/4: 
        print("Pseudo-Euclidean")
    else:
        print("Isotropic")
    # --- Residuals: signed distance of each point to hyperplane ---
    residuals = X @ n + h               # (N,)

    return FitResult(n=n, h=h, kind=kind, signature=sig, residuals=residuals)

def hyperplane_int_space(hyperplane_n, hyperplane_h):
    """
    Hyperplane in R^3,1 given by (n1,n2,n3,n4) and h, corresponds to spheres satisfying n.x + h = 0
    """

    n = hyperplane_n[:3]

    norm_n = np.linalg.norm(n)

    h = hyperplane_h/norm_n

    return n/norm_n, h

def closest_point_to_origin(n: np.ndarray, h: float) -> np.ndarray:
    """
    Returns the point on n.x + h = 0 closest to the origin.
    Works in any dimension — including R^{3,1} cyclographic space.
    
    p0 = -h * n / ||n||^2
    """
    return -h * n / np.dot(n, n)

rs = fit_hyperplane(iso_spheres)

print("Normal n:", rs.n)
print("Origin distance h:", rs.h)
print("Kind:", rs.kind)


# Project iso_spheres to hyperplane 
iso_sph_proj = iso_spheres - np.outer(rs.residuals, rs.n)  # (N,4) - (N,1)*(4,) = (N,4) 

proj_centers = iso_sph_proj[:,:3]
proj_radii = iso_sph_proj[:,3]

# Verify that projected points satisfy n.x + h = 0
#proj_residuals = iso_sph_proj @ rs.n + rs.h

space_plane = hyperplane_int_space(rs.n, rs.h)

space_n = space_plane[0]
space_h = space_plane[1]

# Intersection angle between the hyperplane and R3
phi = np.arccos( np.abs( rs.n @ np.array([0,0,0,1]) ))

cphi = np.abs(  1 / np.tan(phi))


print("angle")
if np.abs(cphi) < 1:
    print("psudo-Euclidean")
    print("cphi", cphi)
    theta = np.arccos( cphi )
elif np.abs(cphi) > 1:
    print("Euclidean")
    theta = np.arccosh( cphi )
else:
    print("Isotropic")
    


radii = 0.8 * np.abs(smallest_radii)

for i in range(len(radii)):
    if cphi < 1:
        # psudo-Euclidean
        radii[i] = (mesh.vertices[i] @ ng + hg)/(np.cos(theta) -  (axes[i] @ ng)/(np.sin(angles[i])))
    elif cphi > 1:
        radii[i] = (mesh.vertices[i] @ ng + hg)/(np.cosh(theta) -  (axes[i] @ ng)/(np.sin(angles[i])))



centers = mesh.vertices + (radii/np.sin(angles))[:, None] * axes


p0 = closest_point_to_origin(space_plane[0], space_plane[1])

print("Space plane n:", space_plane[0])
print("Space plane h:", space_plane[1])
print("plane ",ng ,hg)

draw_plane(p0, space_plane[0], size=(1, 1), name="Fitted_plane", color=(1,0,0), scene=kv)

lmesh = Lmesh()
lmesh.init_L_mesh(
    mesh,
    orientation=1,
    combinatorics="dual",
    mode="SphCongruence",
    radii=radii[inn_idx],
    centers=centers[inn_idx],
)
print( lmesh.compute_oriented_contact_dev())


lmesh.improve_contact()
lmesh.visualize()

sC, sF = lmesh.sphere_center_mesh()

sF_planarity = face_planarity(sC, sF)

kv.register_surface_mesh("Sphere Centers", sC, sF, color=(0.7, 0.9, 0.7), show_edges=True)

kv.add_scalar_quantity("Sphere Centers", "Planairty",defined_on="faces",  values=sF_planarity)

## 5 — Add a vector quantity

Per-vertex vectors are drawn as arrows in the viewer.

In [ ]:
from hanan.geometry.mesh import Mesh
from hanan.geometry.utils import compute_cone_axes

mesh = Mesh()
mesh.make_mesh(V, F)

axes, _ = compute_cone_axes(mesh)

vi, vj = mesh.edge_vertices()

edge_vertices = np.vstack((V[vi], V[vj]))
edges_connection = np.array([[i, i + len(vi)] for i in range(len(vi))])
kv.register_curve_network("Edge Mesh", edge_vertices, edges_connection, color=(0.9, 0.7, 0.7))

kv.add_vector_quantity("Base Mesh", "cone axes", axes, defined_on="vertices")



## 6 — Add a curve network

Register any edge set as a standalone `curve_network` object.

In [ ]:
# Boundary edges of the mesh
boundary_verts  = list(mesh.boundary_vertices())
boundary_edges  = np.array(list(mesh.boundary_edges()), dtype=np.int32)

kv.register_curve_network("Boundary", V, boundary_edges, color=(1.0, 0.3, 0.1))

## 7 — Build and visualize an L-mesh

Uses the same pipeline as `test_subdivision_web.py`.

In [ ]:
from hanan.geometry.utils import conical_mesh_smallest_curvature_sphere
from LMesh.L_mesh import Lmesh
from kayviz import serializers as ser

inn = mesh.inner_vertices()
v_adj = mesh.vertex_adjacency_list()
axes, angles = compute_cone_axes(mesh)
centers_int, smallest_radii = conical_mesh_smallest_curvature_sphere(
    mesh.vertices, axes, angles, v_adj
)

radius  = 0.5 * np.abs(smallest_radii)
centers = V + radius[:, None] * axes

lmesh = Lmesh()
lmesh.init_L_mesh(
    mesh,
    orientation=1,
    combinatorics="dual",
    mode="SphCongruence",
    radii=radius[inn],
    centers=centers[inn],
)
lmesh.improve_contact()

print("L-mesh ready")

In [ ]:
kv.clear()

kv.register_surface_mesh("Base Mesh", V, F, color=(0.5, 0.5, 0.5))
kv.push_objects(ser.lmesh_scene(lmesh, prefix="L-mesh"))

## 8 — Subdivide and update in-place

The viewer scene buffer is replaced each time you call `push_objects` with the same names.

In [ ]:
lmesh.subdivide_lmesh()
lmesh.improve_contact()

kv.push_objects(ser.lmesh_scene(lmesh, prefix="L-mesh"))
print("Subdivision done — viewer updated")

## 9 — Toggle visibility

In [ ]:
kv.set_enabled("Base Mesh", False)   # hide base mesh
# kv.set_enabled("Base Mesh", True)  # show it again

## 10 — Remove an object / clear everything

In [ ]:
kv.remove("Boundary")   # remove one object
# kv.clear()            # remove everything